# Train HelpSteer2 GPT-2 LoRA Adapters

This notebook runs the HelpSteer2 multi-objective training prototype for the thesis pipeline:

$$\delta_i \rightarrow R \rightarrow \lambda = f(p, R) \rightarrow \theta(\lambda)$$

It trains separate GPT-2 LoRA adapters for:

- helpfulness
- correctness
- coherence
- complexity
- verbosity

Each adapter starts from a fresh GPT-2 base model. HelpSteer2 attribute ratings select prompt/response texts for lightweight objective-specific supervised training. The notebook trains and checks all five adapters for later geometry and merging experiments.

## 1. Check GPU

In Colab, select **Runtime > Change runtime type > T4 GPU** before training. A CPU run will be much slower.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training on CPU will be much slower.")

## 2. Clone or update the repository

This cell always starts in `/content`. If `/content/master-thesis/.git` exists, it updates the repository. Otherwise, it clones a fresh copy. If a non-Git folder already uses that path, the cell stops instead of creating `/content/master-thesis/master-thesis`.

In [ ]:
%cd /content

from pathlib import Path
import subprocess

repo_path = Path("/content/master-thesis")

if (repo_path / ".git").is_dir():
    print("Repository already exists. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        "/content/master-thesis exists but is not a Git repository. "
        "Rename or remove that folder, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NZhang137/master-thesis.git"],
        check=True,
    )

%cd /content/master-thesis

## 3. Inspect the repository

Confirm that Colab is in the repository root and that the HelpSteer2 scripts and source utilities are present.

In [ ]:
!pwd
!ls
!ls scripts
!ls src

## 4. Install dependencies

Colab already provides PyTorch. This prototype needs Transformers, Datasets, PEFT, Accelerate, and Safetensors.

Colab may contain an old incompatible `torchao` version, so the installation cell removes it before installing the GPT-2 + LoRA dependencies.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U transformers datasets peft accelerate safetensors

If `torchao` was previously imported or you saw a compatibility error, select **Runtime > Restart session** after the installation cell. Then rerun the GPU, repository, and installation cells.

A future environment that explicitly requires TorchAO can install a current version with:

```python
!pip install -q -U "torchao>=0.16.0"
```

Install it only if another package explicitly requires it.

## 5. Choose one training run

The next two sections write to the same five adapter folders. Run **either** the smoke test **or** the larger prototype run.

Running both is unnecessary: the second run will train the five adapters again and overwrite files in the same output folders.

### Option A: Small smoke test

Use this first when you only want to verify that dataset loading, training, and saving work. The script selects highly rated examples from the first 20 dataset rows and trains all five adapters for one epoch.

In [ ]:
!python scripts/train_helpsteer2_adapters.py --split "train[:20]" --num_epochs 1

### Option B: Slightly larger prototype run

Use this instead of Option A for the default prototype. It considers the first 100 dataset rows and trains all five adapters for one epoch.

In [ ]:
!python scripts/train_helpsteer2_adapters.py --split "train[:100]" --num_epochs 1

## 6. Check adapter files

After your selected training run finishes, verify that all five output folders contain `adapter_config.json` and `adapter_model.safetensors`.

In [ ]:
!python scripts/check_helpsteer2_adapters.py
!ls adapters

The expected folders are:

- `adapters/helpsteer2-gpt2-helpfulness-adapter`
- `adapters/helpsteer2-gpt2-correctness-adapter`
- `adapters/helpsteer2-gpt2-coherence-adapter`
- `adapters/helpsteer2-gpt2-complexity-adapter`
- `adapters/helpsteer2-gpt2-verbosity-adapter`

## 7. Zip adapters for local backup

**Warning:** `helpsteer2_adapters.zip` is only a local backup for downloading from Colab. The ZIP and the generated adapter folders must not be committed to GitHub.

In [ ]:
!zip -r helpsteer2_adapters.zip adapters/

Download `helpsteer2_adapters.zip` from the Colab file browser if you want to keep the trained adapters after the runtime ends. Do not add the ZIP to the repository.

## 8. Git safety check

The generated files should remain local and be ignored by Git. Do **not** commit:

- `adapters/`
- `helpsteer2_adapters.zip`
- `.safetensors` or `.bin` files
- checkpoints or full model files

In [ ]:
!git status

## What this notebook establishes

This notebook checks whether five objective-specific HelpSteer2 LoRA adapters can be trained independently from the same fresh GPT-2 base model and saved for later merging experiments.